# De la primera pantalla al pago

**Brenis Hernández · Proyecto de portafolio**

Esta copia pública conserva el código y las explicaciones. Se retiraron las salidas originales para evitar publicar registros o identificadores; los resultados resumidos se documentan en la ficha del portafolio a partir de la versión original revisada. No se ha vuelto a ejecutar sobre los datos originales. También se omiten mensajes de revisión académica.

**Alcance:** El 18.33% depende del orden de las primeras ocurrencias y no es la proporción global de usuarios que pagaron: 3,539 de 7,534 usuarios registraron un pago (46.97%). Los usuarios pueden repetir pantallas o seguir otro recorrido. No rechazar una hipótesis no demuestra equivalencia entre variantes.


# Sprint 12 – Proyecto

## Introducción
Trabajamos en una empresa emergente que vende productos alimenticios. El objetivo es investigar el comportamiento del usuario dentro de la aplicación.

Este proyecto se divide en dos partes:
1. **Estudiar el embudo de ventas**: identificar cómo avanzan los usuarios hacia la compra y en qué etapas se pierden.
2. **Analizar un experimento A/A/B**: evaluar si un cambio de tipografía (grupo 248) afecta el comportamiento de los usuarios comparado con los grupos de control (246 y 247).



## Datos
Cada registro representa un evento generado por un usuario:
- **EventName**: nombre del evento.
- **DeviceIDHash**: identificador único del usuario.
- **EventTimestamp**: momento del evento.
- **ExpId**: grupo experimental (246 y 247 = control; 248 = prueba).


## Paso 1. Abrir el archivo de datos y leer la información general

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from math import sqrt

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

path = '/datasets/logs_exp_us.csv'
df = pd.read_csv(path, sep='\t')

df.head()



En esta sección reviso la estructura del dataset, tipos de datos, valores ausentes y ejemplos de registros.


In [ ]:
df.info()


In [ ]:
df.isna().sum()


In [ ]:
df.duplicated().sum()


In [ ]:
df = df.drop_duplicates().reset_index(drop=True)


Se encontraron 413 registros duplicados. Para evitar sesgos en el conteo de eventos, se eliminaron duplicados antes de continuar con el análisis.


## Paso 2. Preparar los datos para el análisis
- Renombro columnas para mayor comodidad.
- Convierto el timestamp a datetime.
- Creo una columna de fecha (sin hora).
- Reviso tipos de datos y valores faltantes.


In [ ]:
df = df.rename(columns={
    'EventName': 'event',
    'DeviceIDHash': 'user_id',
    'EventTimestamp': 'ts',
    'ExpId': 'exp_id'
})

df['datetime'] = pd.to_datetime(df['ts'], unit='s')
df['date'] = df['datetime'].dt.date

df.head()


Validacion de tipos 

In [ ]:
df.dtypes


## Paso 3. Estudiar y comprobar los datos

Responderé:
- ¿Cuántos eventos hay?
- ¿Cuántos usuarios hay?
- ¿Promedio de eventos por usuario?
- ¿Qué periodo de tiempo cubren los datos?
- ¿Perdiste muchos eventos y usuarios al excluir los datos más antiguos?
- Definiré un corte de "datos completos" y excluiré lo anterior.
- Verificaré que existan usuarios en los 3 grupos experimentales.


In [ ]:
# Código: eventos, usuarios, promedio
total_events = len(df)
total_users = df['user_id'].nunique()
avg_events_per_user = total_events / total_users

total_events, total_users, avg_events_per_user


In [ ]:
# Código: rango de fechas
min_dt = df['datetime'].min()
max_dt = df['datetime'].max()
min_dt, max_dt


In [ ]:
# Codigo: histograma/barras por fecha
daily_events = df.groupby('date').size().sort_index()

plt.figure(figsize=(12,4))
daily_events.plot(kind='bar')
plt.title('Eventos por día')
plt.xlabel('Fecha')
plt.ylabel('Número de eventos')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


### Determinación de periodo "completo"

Al analizar el número de eventos por día, se observa que durante el periodo del **25 al 31 de julio de 2019** el volumen de eventos es muy bajo e irregular. Esto sugiere que los datos de esos días están incompletos y podrían sesgar el análisis.

A partir del **1 de agosto de 2019**, el número de eventos diarios aumenta considerablemente y se mantiene relativamente estable, lo que indica que los datos son más completos y representativos del comportamiento real de los usuarios.

Por esta razón, para evitar sesgos en el análisis posterior, se decidió **excluir los registros anteriores al 1 de agosto de 2019** y trabajar únicamente con el periodo completo.



In [ ]:
cut_date = pd.to_datetime('2019-08-01').date()

df_before = df.copy()
df = df[df['date'] >= cut_date].copy()

print("Antes del recorte:")
print("Eventos:", len(df_before), "Usuarios:", df_before['user_id'].nunique())

print("\nDespués del recorte:")
print("Eventos:", len(df), "Usuarios:", df['user_id'].nunique())

print("\nPerdidos:")
print("Eventos perdidos:", len(df_before) - len(df))
print("Usuarios perdidos:", df_before['user_id'].nunique() - df['user_id'].nunique())


In [ ]:
# Código: verificar grupos
users_by_group = df.groupby('exp_id')['user_id'].nunique().sort_index()
users_by_group


## Paso 4. Embudo de eventos

Haré:
1. Listado de eventos y su frecuencia.
2. Usuarios únicos que realizaron cada evento.
3. Proporción de usuarios por evento.
4. Definir el orden del embudo (eventos principales) y calcular:
   - conversión entre etapas
   - etapa con mayor pérdida
   - % que completa todo el recorrido hasta el pago





### Observa qué eventos hay en los registros y su frecuencia de suceso. Ordénalos por frecuencia.

In [ ]:
event_counts = df['event'].value_counts()
event_counts





### Encuentra la cantidad de usuarios que realizaron cada una de estas acciones. Ordena los eventos por el número de usuarios. Calcula la proporción de usuarios que realizaron la acción al menos una vez.

In [ ]:
users_per_event = df.groupby('event')['user_id'].nunique().sort_values(ascending=False)
share_users_per_event = (users_per_event / df['user_id'].nunique()).sort_values(ascending=False)

users_per_event, share_users_per_event


### Orden del embudo
Con base en los eventos más relevantes, usaré la secuencia:
**MainScreenAppear → OffersScreenAppear → CartScreenAppear → PaymentScreenSuccessful**




In [ ]:
funnel_events = ['MainScreenAppear','OffersScreenAppear','CartScreenAppear','PaymentScreenSuccessful']

# Primera vez que cada usuario hace cada evento
first_times = (
    df[df['event'].isin(funnel_events)]
    .sort_values('datetime')
    .groupby(['user_id','event'])['datetime'].min()
    .unstack()
)

# Máscaras secuenciales
m1 = first_times['MainScreenAppear'].notna()
m2 = m1 & first_times['OffersScreenAppear'].notna() & (first_times['OffersScreenAppear'] >= first_times['MainScreenAppear'])
m3 = m2 & first_times['CartScreenAppear'].notna() & (first_times['CartScreenAppear'] >= first_times['OffersScreenAppear'])
m4 = m3 & first_times['PaymentScreenSuccessful'].notna() & (first_times['PaymentScreenSuccessful'] >= first_times['CartScreenAppear'])

funnel_counts = pd.Series({
    'MainScreenAppear': m1.sum(),
    'OffersScreenAppear': m2.sum(),
    'CartScreenAppear': m3.sum(),
    'PaymentScreenSuccessful': m4.sum()
})

funnel_counts




### ¿En qué etapa pierdes más usuarios?

In [ ]:
conv_main_offers = funnel_counts['OffersScreenAppear'] / funnel_counts['MainScreenAppear']
conv_offers_cart = funnel_counts['CartScreenAppear'] / funnel_counts['OffersScreenAppear']
conv_cart_payment = funnel_counts['PaymentScreenSuccessful'] / funnel_counts['CartScreenAppear']
conv_total = funnel_counts['PaymentScreenSuccessful'] / funnel_counts['MainScreenAppear']

conv_main_offers, conv_offers_cart, conv_cart_payment, conv_total


In [ ]:
losses = pd.Series({
    'Main→Offers': 1 - conv_main_offers,
    'Offers→Cart': 1 - conv_offers_cart,
    'Cart→Payment': 1 - conv_cart_payment
}).sort_values(ascending=False)

losses


In [ ]:
plt.figure(figsize=(8,4))
funnel_counts.plot(kind='bar')
plt.title('Embudo secuencial: usuarios por etapa')
plt.xlabel('Etapa')
plt.ylabel('Usuarios')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


### Conclusiones del embudo


Al analizar los registros de eventos, se identificaron cinco acciones principales dentro de la aplicación: 
`MainScreenAppear`, `OffersScreenAppear`, `CartScreenAppear`, `PaymentScreenSuccessful` y `Tutorial`.  
Entre estos, el evento más frecuente es **MainScreenAppear**, lo cual es consistente con el hecho de que representa el punto de entrada principal de los usuarios a la aplicación. Le siguen los eventos relacionados con la exploración de ofertas, el acceso al carrito y, finalmente, el pago. El evento `Tutorial` ocurre con mucha menor frecuencia, lo que indica que solo una parte de los usuarios interactúa con esta funcionalidad.

Al observar la cantidad de usuarios que realizaron cada acción al menos una vez, se confirma que casi todos los usuarios llegan a la pantalla principal, pero el número disminuye progresivamente en las etapas posteriores. Aproximadamente el **61%** de los usuarios visualiza la pantalla de ofertas, alrededor del **50%** llega al carrito y cerca del **47%** alcanza la pantalla de pago. Esto muestra una pérdida gradual de usuarios conforme avanzan en el proceso de compra.

Con base en el orden temporal de los eventos, se determinó que las acciones principales siguen la secuencia lógica:
**MainScreenAppear → OffersScreenAppear → CartScreenAppear → PaymentScreenSuccessful**.  
No todos los eventos forman parte de una única secuencia obligatoria, ya que, por ejemplo, el evento `Tutorial` puede ocurrir en distintos momentos o no ocurrir en absoluto para muchos usuarios. Por esta razón, el tutorial no se incluyó en el cálculo del embudo principal.

Al analizar el embudo secuencial, se encontró que:
- La conversión de **MainScreenAppear a OffersScreenAppear** es de aproximadamente **56.6%**.
- La conversión de **OffersScreenAppear a CartScreenAppear** es de aproximadamente **42.7%**.
- La conversión de **CartScreenAppear a PaymentScreenSuccessful** es de aproximadamente **75.7%**.

La mayor pérdida de usuarios ocurre en la transición de **OffersScreenAppear a CartScreenAppear**, lo que sugiere que esta etapa representa el principal punto de fricción dentro del embudo de ventas.

Finalmente, al evaluar el recorrido completo desde el primer evento hasta el pago, se observó que solo alrededor del **18.3%** de los usuarios completa todo el proceso desde la pantalla principal hasta la confirmación del pago. Esto indica que, aunque una parte importante de los usuarios muestra interés inicial en los productos, solo una minoría logra completar la compra.


## Paso 5. Estudiar los resultados del experimento

Objetivo: evaluar si el cambio de tipografía (grupo 248) afecta la actividad de los usuarios.

Haré:
1. Usuarios por grupo.
2. Comparación A/A: 246 vs 247.
3. Comparación A/B: 248 vs 246, 248 vs 247 y 248 vs (246+247).
4. Ajuste por pruebas múltiples (Bonferroni).
5. Conclusión final.


In [ ]:
users_by_group


In [ ]:
group_users = df.groupby('exp_id')['user_id'].nunique()
user_event = df.groupby(['exp_id','event'])['user_id'].nunique().unstack(fill_value=0)

group_users, user_event.head()


In [ ]:
# función z-test para proporciones
def prop_z_test(success_a, n_a, success_b, n_b):
    """
    Prueba z de 2 proporciones.
    success_a: usuarios (éxitos) en grupo A
    n_a: total usuarios grupo A
    success_b: usuarios (éxitos) en grupo B
    n_b: total usuarios grupo B
    """
    p1 = success_a / n_a
    p2 = success_b / n_b
    p_pool = (success_a + success_b) / (n_a + n_b)

    se = sqrt(p_pool * (1 - p_pool) * (1/n_a + 1/n_b))
    if se == 0:
        return 0, 1.0, p1, p2

    z = (p1 - p2) / se
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))
    return z, p_value, p1, p2


### A/A Test: 246 vs 247
Primero verifico que los dos grupos de control sean estadísticamente similares.
Si hubiera diferencias significativas, podría indicar un problema en la división de grupos o sesgo en los datos.


In [ ]:
# correr pruebas para todos los eventos
aa_results = []
for ev in user_event.columns:
    z, p, p246, p247 = prop_z_test(
        user_event.loc[246, ev], group_users.loc[246],
        user_event.loc[247, ev], group_users.loc[247]
    )
    aa_results.append([ev, p246, p247, p])

aa_results = pd.DataFrame(aa_results, columns=['event', 'share_246', 'share_247', 'p_value']).sort_values('p_value')
aa_results


**Interpretación A/A:**
- Si la mayoría de p-values > 0.05, los grupos 246 y 247 son equivalentes.
- Si aparecen p-values pequeños, revisaría posibles sesgos o un problema de asignación.


### A/B Test: grupo 248 (fuente nueva) vs controles
Comparo el grupo de prueba contra:
- Control 246
- Control 247
- Controles combinados (246+247)

Esto permite evaluar si la tipografía genera cambios en la proporción de usuarios que realizan cada evento.


In [ ]:
combined_n = group_users.loc[246] + group_users.loc[247]
combined_counts = user_event.loc[246] + user_event.loc[247]


In [ ]:
ab_rows = []
for ev in user_event.columns:
    # 248 vs 246
    _, p_248_246, s248, s246 = prop_z_test(
        user_event.loc[248, ev], group_users.loc[248],
        user_event.loc[246, ev], group_users.loc[246]
    )
    # 248 vs 247
    _, p_248_247, s248b, s247 = prop_z_test(
        user_event.loc[248, ev], group_users.loc[248],
        user_event.loc[247, ev], group_users.loc[247]
    )
    # 248 vs combinados
    _, p_248_comb, s248c, scomb = prop_z_test(
        user_event.loc[248, ev], group_users.loc[248],
        combined_counts[ev], combined_n
    )
    ab_rows.append([ev, s248, s246, p_248_246, s247, p_248_247, scomb, p_248_comb])

ab_results = pd.DataFrame(
    ab_rows,
    columns=['event', 'share_248', 'share_246', 'p_248_vs_246', 'share_247', 'p_248_vs_247', 'share_controls', 'p_248_vs_controls']
).sort_values('p_248_vs_controls')

ab_results


### Corrección por pruebas múltiples (Bonferroni)
Al probar muchos eventos/comparaciones, aumenta el riesgo de falsos positivos.

Calculo:
- Número total de pruebas (m)
- Alfa inicial (α)
- Alfa corregida: α_corr = α / m

Luego verifico qué comparaciones siguen siendo significativas con α_corr.


In [ ]:
alpha = 0.1

# Conteo de pruebas: aquí estamos evaluando p_248_vs_246, p_248_vs_247 y p_248_vs_controls para cada evento
m = len(user_event.columns) * 3
alpha_corr = alpha / m

m, alpha_corr


In [ ]:
ab_results['sig_alpha_0_05_controls'] = ab_results['p_248_vs_controls'] < 0.05
ab_results['sig_alpha_0_1_controls'] = ab_results['p_248_vs_controls'] < 0.1
ab_results['sig_bonferroni_controls'] = ab_results['p_248_vs_controls'] < alpha_corr

ab_results[['event','p_248_vs_controls','sig_alpha_0_05_controls','sig_alpha_0_1_controls','sig_bonferroni_controls']].sort_values('p_248_vs_controls')


## Conclusiones del experimento

### A/A (246 vs 247)
Al comparar los dos grupos de control (246 y 247), se analizaron las proporciones de usuarios que realizaron cada uno de los eventos principales de la aplicación. En todos los casos, los valores p obtenidos fueron mayores al nivel de significancia estándar (α = 0.05), lo que indica que **no existen diferencias estadísticamente significativas** entre ambos grupos.

Esto confirma que los grupos de control son comparables y que la asignación de usuarios a los grupos experimentales se realizó correctamente, permitiendo continuar con el análisis A/B de forma confiable.

### A/B (248 vs grupos de control)
Posteriormente, se comparó el grupo de prueba (248), que utilizó la nueva tipografía, contra cada uno de los grupos de control (246 y 247) y contra los grupos de control combinados. Para cada evento se evaluaron diferencias en la proporción de usuarios que realizaron la acción.

En comparaciones aisladas (por ejemplo, **248 vs 246 para `CartScreenAppear`**) puede observarse algún valor p relativamente menor; sin embargo, al considerar el conjunto de pruebas realizadas y aplicar corrección por pruebas múltiples (Bonferroni), **ninguna diferencia se mantiene como estadísticamente significativa**, por lo que estos resultados pueden atribuirse al azar.

### Decisión
Con los resultados actuales, **no hay evidencia estadística suficiente** para afirmar que el cambio de tipografía (grupo 248) afecte el comportamiento de los usuarios. Por lo tanto, no se recomienda tomar la decisión de cambiar las fuentes basándose únicamente en este experimento.

### Recomendaciones
- Mantener el experimento por más tiempo o aumentar el tamaño de la muestra para mejorar el poder estadístico.
- Considerar métricas adicionales (por ejemplo: tiempo al pago, número de sesiones por usuario o ingresos por usuario, si estuvieran disponibles).
- Priorizar mejoras en el embudo, especialmente en la transición **OffersScreenAppear → CartScreenAppear**, donde se observa la mayor pérdida de usuarios.


## Conclusión general

En este proyecto se analizó el comportamiento de los usuarios dentro de una aplicación de venta de productos alimenticios, combinando el estudio del embudo de ventas con la evaluación de un experimento A/A/B relacionado con un cambio de tipografía en la interfaz.

A partir del análisis exploratorio inicial, se identificó que los datos anteriores al **1 de agosto de 2019** eran incompletos y podían sesgar los resultados. Tras excluir ese periodo, se trabajó con un conjunto de datos estable y representativo, que permitió realizar un análisis confiable tanto del embudo como del experimento.

El análisis del embudo de eventos mostró que la mayoría de los usuarios accede a la pantalla principal de la aplicación, pero existe una pérdida progresiva de usuarios en las etapas posteriores. La mayor caída ocurre en la transición de **OffersScreenAppear a CartScreenAppear**, lo que sugiere que esta etapa representa el principal punto de fricción en el proceso de compra. Aunque la conversión de carrito a pago es relativamente alta, solo alrededor del **18% de los usuarios** completa todo el recorrido desde la pantalla principal hasta el pago, lo que indica un amplio margen de mejora en la experiencia de usuario y en la optimización del flujo de compra.

En la segunda parte del proyecto, se evaluó un experimento A/A/B para determinar si un cambio en la tipografía de la aplicación afectaba el comportamiento de los usuarios. La comparación entre los dos grupos de control (246 y 247) no mostró diferencias estadísticamente significativas, lo que confirma que la división de los usuarios fue adecuada. Posteriormente, al comparar el grupo de prueba (248) con los grupos de control, no se encontraron diferencias estadísticamente significativas en las proporciones de usuarios que realizaron los eventos clave, especialmente al aplicar correcciones por pruebas múltiples.

Con base en los resultados obtenidos, no existe evidencia estadística suficiente para afirmar que el cambio de tipografía tenga un impacto significativo en el comportamiento de los usuarios o en el embudo de conversión. Por lo tanto, la decisión de implementar el nuevo diseño no debería basarse únicamente en este experimento.

En conclusión, el análisis permitió identificar claramente los puntos críticos del embudo de ventas y validar la correcta ejecución del experimento A/A/B. Para futuras mejoras, se recomienda enfocar los esfuerzos en optimizar la transición entre la pantalla de ofertas y el carrito, así como ampliar o prolongar los experimentos para obtener resultados más robustos y evaluar métricas adicionales relacionadas con la experiencia del usuario.
